# 02 — Inspect Raw Data

**Purpose:** Examine FITS headers and plot raw FLT images for all 6 quasars to verify
data quality before committing to processing.

**Inputs:** FLT files in `data/raw/<quasar>/`

**Outputs:** Diagnostic plots and a header summary table

**Next step:** `03_align_images.ipynb`

## Imports

In [ ]:
import os
import glob
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.visualization import ZScaleInterval

## Define paths

In [ ]:
RAW_DIR = Path('../data/raw')
quasar_dirs = sorted([d for d in RAW_DIR.iterdir() if d.is_dir()])
print(f'Quasar directories found: {[d.name for d in quasar_dirs]}')

## Header summary

In [ ]:
for qdir in quasar_dirs:
    flt_files = sorted(qdir.glob('*flt.fits'))
    print(f'\n{qdir.name} — {len(flt_files)} FLT files')
    for f in flt_files:
        with fits.open(f) as hdul:
            hdr = hdul[0].header
            print(f"  {f.name}  filter={hdr.get('FILTER','?')}  exptime={hdr.get('EXPTIME','?')}s")

## Plot raw images

In [ ]:
zscale = ZScaleInterval()

for qdir in quasar_dirs:
    flt_files = sorted(qdir.glob('*flt.fits'))
    if not flt_files:
        print(f'{qdir.name}: no FLT files found')
        continue
    
    n = len(flt_files)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1:
        axes = [axes]
    fig.suptitle(qdir.name, fontsize=12)
    
    for ax, f in zip(axes, flt_files):
        with fits.open(f) as hdul:
            data = hdul['SCI', 1].data
        vmin, vmax = zscale.get_limits(data)
        ax.imshow(data, vmin=vmin, vmax=vmax, origin='lower', cmap='gray')
        ax.set_title(f.name, fontsize=7)
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

## Next steps

Review the plots and header summary above. If all 6 quasars look reasonable,
proceed to `03_align_images.ipynb`. Flag any exposures with obvious artifacts
using `/log-issue` before continuing.